# SafeSpan Bridge Analytics: Predictive Maintenance for U.S. Bridge Infrastructure
## Final Presentation Notebook — DATA 245

**Authors:** SafeSpan Team
**Dataset:** National Bridge Inventory (NBI), 2018–2025
**Target:** 4-class bridge condition (Critical / Poor / Fair / Good)

---

### What this notebook covers

This notebook implements three analytical extensions on top of the SafeSpan
classification pipeline:

| Module | Purpose |
|--------|---------|
| **Data Drift** | Detect whether NBI feature distributions shift across inspection years |
| **Model Performance Drift** | Evaluate whether a temporally-trained model stays reliable on future years |
| **Survival Analysis** | Estimate *how long* until a bridge deteriorates into Poor/Critical condition |

Together, these transform SafeSpan from a static snapshot classifier into a
**temporal maintenance decision-support framework**.


---
## Section 0 — Prerequisites

Install any missing packages before running:
```bash
pip install lightgbm lifelines imbalanced-learn pyarrow
```


In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys

# Use non-interactive backend only when running as a plain .py script.
# In Jupyter the kernel sets its own inline backend — don't override it.
import matplotlib
try:
    get_ipython()  # noqa: F821 — defined inside Jupyter, NameError outside
except NameError:
    matplotlib.use("Agg")  # headless rendering for script mode

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.spatial.distance import jensenshannon

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# LightGBM (optional — falls back to RandomForest if not installed)
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
    print("✅ LightGBM available")
except Exception:
    LGBM_AVAILABLE = False
    print("⚠️  LightGBM not available — will use RandomForest as fallback")
    print("    Install with: pip install lightgbm")

# Lifelines for survival analysis
try:
    from lifelines import KaplanMeierFitter, CoxPHFitter
    from lifelines.statistics import logrank_test
    LIFELINES_AVAILABLE = True
    print("✅ lifelines available")
except ImportError:
    LIFELINES_AVAILABLE = False
    print("⚠️  lifelines not found — survival analysis sections will be skipped")
    print("    Install with: pip install lifelines")

# ─── Plot style ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "figure.figsize": (12, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

CONDITION_COLORS = {
    "Critical": "#e74c3c",
    "Poor":     "#e67e22",
    "Fair":     "#f1c40f",
    "Good":     "#27ae60",
}
CLASS_ORDER = ["Critical", "Poor", "Fair", "Good"]

print("\n✅ All core libraries loaded successfully")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.11/site-p

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.11/site-p

AttributeError: _ARRAY_API not found

✅ LightGBM available
⚠️  lifelines not found — survival analysis sections will be skipped
    Install with: pip install lifelines

✅ All core libraries loaded successfully


---
## Section 1 — Data Loading & Column Detection

The notebook tries to load `safe_span_cleaned.csv` first.
If that file is absent it falls back to `nbi_cleaned.parquet`, then `nbi_cleaned.csv`.
If none exist it falls back to the raw `nbi_5million.csv` and re-applies the
cleaning / feature-engineering pipeline inline.


In [2]:
# ─── File search order ────────────────────────────────────────────────────────
_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
_PROJECT_ROOT = os.path.abspath(os.path.join(_SCRIPT_DIR, "..", "..", ".."))  # up from worktree

CANDIDATE_FILES = [
    os.path.join(_SCRIPT_DIR,    "safe_span_cleaned.csv"),
    os.path.join(_PROJECT_ROOT,  "safe_span_cleaned.csv"),
    os.path.join(_PROJECT_ROOT,  "nbi_cleaned.parquet"),
    os.path.join(_PROJECT_ROOT,  "nbi_cleaned.csv"),
    os.path.join(_PROJECT_ROOT,  "nbi_5million.csv"),
]

# Max rows to load — keeps memory usage under ~2 GB for the analysis sections.
# Increase this value if your machine has more RAM (e.g. 1_000_000 for 16 GB+).
GLOBAL_ROW_LIMIT = 300_000

def _load_any(candidates, row_limit=None):
    """Try candidate paths in order; return (df, path) for the first that loads."""
    for path in candidates:
        if not os.path.exists(path):
            continue
        ext = os.path.splitext(path)[1].lower()
        try:
            if ext == ".parquet":
                df = pd.read_parquet(path)
            else:
                df = pd.read_csv(path, low_memory=False)

            # Stratified sample if dataset is large
            if row_limit and len(df) > row_limit and "TARGET_CONDITION" in df.columns:
                df = (
                    df[df["TARGET_CONDITION"].isin(["Critical","Poor","Fair","Good"])]
                      .groupby("TARGET_CONDITION", group_keys=False)
                      .apply(lambda g: g.sample(
                          frac=row_limit / len(df), random_state=42
                      ))
                      .sample(frac=1, random_state=42)
                      .reset_index(drop=True)
                )
                print(f"ℹ️  Sampled to {len(df):,} rows (stratified by condition)")
            elif row_limit and len(df) > row_limit:
                df = df.sample(row_limit, random_state=42).reset_index(drop=True)
                print(f"ℹ️  Randomly sampled to {len(df):,} rows")

            print(f"✅ Loaded: {path}  ({df.shape[0]:,} rows × {df.shape[1]} cols)")
            return df, path
        except Exception as exc:
            print(f"⚠️  Could not read {path}: {exc}")
    return None, None

df_raw, DATA_SOURCE = _load_any(CANDIDATE_FILES, row_limit=GLOBAL_ROW_LIMIT)

if df_raw is None:
    raise FileNotFoundError(
        "No source file found. Please place safe_span_cleaned.csv in the same "
        "directory as this notebook, or run the EDA/data-collection notebooks first."
    )


ℹ️  Sampled to 92,408 rows (stratified by condition)
✅ Loaded: /Users/akash/dev/DATA245/Project/nbi_cleaned.parquet  (92,408 rows × 127 cols)


### 1.1 — Feature Engineering (runs only when loading raw / uncleaned data)

If we loaded the pre-engineered cleaned file the block below detects that the
engineered columns already exist and skips re-creation.


In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Apply SafeSpan domain feature engineering."""
    df = df.copy()

    # ── TARGET_CONDITION ──────────────────────────────────────────────────────
    if "TARGET_CONDITION" not in df.columns:
        def _categorize(row):
            try:
                ratings = [
                    pd.to_numeric(row.get(c, np.nan), errors="coerce")
                    for c in ["DECK_COND_058", "SUPERSTRUCTURE_COND_059",
                              "SUBSTRUCTURE_COND_060", "CULVERT_COND_062"]
                ]
                valid = [r for r in ratings if pd.notna(r) and r <= 9]
                if not valid:
                    return "Unknown"
                m = min(valid)
                if m <= 3:   return "Critical"
                if m == 4:   return "Poor"
                if m <= 6:   return "Fair"
                return "Good"
            except Exception:
                return "Unknown"
        df["TARGET_CONDITION"] = df.apply(_categorize, axis=1)

    df = df[df["TARGET_CONDITION"].isin(CLASS_ORDER)].copy()

    # ── Fix SCOUR_CRITICAL_113 ─────────────────────────────────────────────────
    if "SCOUR_CRITICAL_113" in df.columns and df["SCOUR_CRITICAL_113"].dtype == object:
        df["SCOUR_CRITICAL_113"] = pd.to_numeric(
            df["SCOUR_CRITICAL_113"].replace({"N": np.nan, "U": np.nan, "T": np.nan}),
            errors="coerce",
        )

    # ── Engineered features ───────────────────────────────────────────────────
    if "YEAR_BUILT_027" in df.columns and "YEAR" in df.columns:
        if "BRIDGE_AGE" not in df.columns:
            df["BRIDGE_AGE"] = df["YEAR"] - df["YEAR_BUILT_027"]

    if "BRIDGE_AGE" in df.columns and "MAX_SPAN_LEN_MT_048" in df.columns:
        if "AGE_TO_SPAN_RATIO" not in df.columns:
            df["AGE_TO_SPAN_RATIO"] = (
                df["BRIDGE_AGE"] / df["MAX_SPAN_LEN_MT_048"].replace(0, np.nan)
            )

    if "ADT_029" in df.columns and "STRUCTURE_LEN_MT_049" in df.columns:
        if "TRAFFIC_DENSITY" not in df.columns:
            df["TRAFFIC_DENSITY"] = (
                df["ADT_029"] / df["STRUCTURE_LEN_MT_049"].replace(0, np.nan)
            )

    if "DECK_WIDTH_MT_052" in df.columns and "ROADWAY_WIDTH_MT_051" in df.columns:
        if "DECK_TO_ROADWAY_RATIO" not in df.columns:
            df["DECK_TO_ROADWAY_RATIO"] = (
                df["DECK_WIDTH_MT_052"] / df["ROADWAY_WIDTH_MT_051"].replace(0, np.nan)
            )

    if "FUTURE_ADT_114" in df.columns and "ADT_029" in df.columns:
        if "ADT_GROWTH_RATIO" not in df.columns:
            df["ADT_GROWTH_RATIO"] = (
                df["FUTURE_ADT_114"] / df["ADT_029"].replace(0, np.nan)
            )

    if "YEAR_RECONSTRUCTED_106" in df.columns:
        if "WAS_RECONSTRUCTED" not in df.columns:
            df["WAS_RECONSTRUCTED"] = (df["YEAR_RECONSTRUCTED_106"] > 0).astype(int)
        if "TIME_SINCE_RECONSTRUCTION" not in df.columns and "BRIDGE_AGE" in df.columns:
            df["TIME_SINCE_RECONSTRUCTION"] = np.where(
                df["YEAR_RECONSTRUCTED_106"] > 0,
                df["YEAR"] - df["YEAR_RECONSTRUCTED_106"],
                df["BRIDGE_AGE"],
            )

    if "BRIDGE_AGE" in df.columns and "SCOUR_CRITICAL_113" in df.columns:
        if "AGE_X_SCOUR" not in df.columns:
            df["AGE_X_SCOUR"] = df["BRIDGE_AGE"] * df["SCOUR_CRITICAL_113"]

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df


ALREADY_ENGINEERED = all(
    c in df_raw.columns
    for c in ["BRIDGE_AGE", "TRAFFIC_DENSITY", "TARGET_CONDITION"]
)

if ALREADY_ENGINEERED:
    df = df_raw.copy()
    print("✅ Engineered features already present — skipping re-engineering")
else:
    print("⚙️  Applying feature engineering pipeline …")
    df = engineer_features(df_raw)
    print(f"✅ Feature engineering complete. Shape: {df.shape}")


⚙️  Applying feature engineering pipeline …
✅ Feature engineering complete. Shape: (0, 128)


### 1.2 — Column Detection & Dataset Summary


In [4]:
# ─── Auto-detect key columns ──────────────────────────────────────────────────
def detect_col(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            print(f"  ✅ {label}: '{c}'")
            return c
    print(f"  ⚠️  {label}: not found (tried {candidates})")
    return None

print("=== Column Detection ===")
YEAR_COL   = detect_col(df, ["YEAR", "INSPECTION_YEAR", "YEAR_BUILT_027"], "Year")
BRIDGE_COL = detect_col(df, ["STRUCTURE_NUMBER_008", "BRIDGE_ID", "OTHR_STATE_STRUC_NO_099"], "Bridge ID")
TARGET_COL = detect_col(df, ["TARGET_CONDITION"], "Target")

if TARGET_COL is None:
    raise ValueError("TARGET_CONDITION column is required but not found.")

# ─── Dataset overview ─────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f" DATASET OVERVIEW")
print(f"{'='*55}")
print(f"  Rows:       {len(df):>12,}")
print(f"  Columns:    {df.shape[1]:>12,}")
if YEAR_COL:
    years = sorted(df[YEAR_COL].dropna().unique())
    print(f"  Years:      {[int(y) for y in years]}")
print(f"\n  Target distribution:")
tc = df[TARGET_COL].value_counts().reindex(CLASS_ORDER)
for cond in CLASS_ORDER:
    n = tc.get(cond, 0)
    pct = n / len(df) * 100
    print(f"    {cond:<10s}: {n:>10,}  ({pct:5.2f}%)")
print(f"{'='*55}")


=== Column Detection ===
  ✅ Year: 'YEAR'
  ✅ Bridge ID: 'OTHR_STATE_STRUC_NO_099'
  ✅ Target: 'TARGET_CONDITION'

 DATASET OVERVIEW
  Rows:                  0
  Columns:             128
  Years:      []

  Target distribution:
    Critical  :        nan  (  nan%)
    Poor      :        nan  (  nan%)
    Fair      :        nan  (  nan%)
    Good      :        nan  (  nan%)


In [5]:
# ─── Quick overview plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart — absolute counts
counts = df[TARGET_COL].value_counts().reindex(CLASS_ORDER)
bars = axes[0].bar(
    CLASS_ORDER, counts.values,
    color=[CONDITION_COLORS[c] for c in CLASS_ORDER],
    edgecolor="black", linewidth=0.7
)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() * 1.01, f"{val:,}",
                 ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[0].set_title("Bridge Condition Class Distribution", fontweight="bold")
axes[0].set_ylabel("Number of Bridges")

# Pie chart
axes[1].pie(
    counts.values, labels=CLASS_ORDER,
    colors=[CONDITION_COLORS[c] for c in CLASS_ORDER],
    autopct="%1.1f%%", startangle=90,
    explode=[0.08, 0.04, 0, 0],
)
axes[1].set_title("Condition Share (%)", fontweight="bold")

plt.suptitle("SafeSpan — NBI Dataset Summary", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/00_dataset_overview.png", dpi=150, bbox_inches="tight")
plt.show()


ValueError: cannot convert float NaN to integer

ValueError: need at least one array to concatenate

<Figure size 1690x520 with 2 Axes>

---
## Section 2 — Data Drift Analysis

**Goal:** Detect whether NBI feature distributions shift across inspection years.

A model trained on Year *T* data may underperform on Year *T+k* data if the
underlying feature distributions have changed — this is called **data drift**.

### Why it matters for SafeSpan
Bridge inventories change each year: bridges are built, reconstructed, or
removed; inspection frequencies change; state reporting practices evolve.
If these changes cause feature distributions to drift, the LightGBM model
trained on 2018–2022 data may be less accurate on 2024–2025 inspections.

### Metrics used
| Metric | Features | Interpretation |
|--------|----------|----------------|
| **Population Stability Index (PSI)** | Numeric | < 0.10 stable · 0.10–0.25 moderate · ≥ 0.25 significant |
| **Jensen-Shannon Distance** | Categorical | < 0.05 low · 0.05–0.15 moderate · > 0.15 high |


In [ ]:
# ─── PSI helper ──────────────────────────────────────────────────────────────
def compute_psi(base_series: pd.Series, comp_series: pd.Series,
                n_bins: int = 10) -> float:
    """
    Population Stability Index between two numeric distributions.
    Returns np.nan if the calculation cannot be performed.
    """
    base = base_series.dropna().values
    comp = comp_series.dropna().values

    if len(base) < 30 or len(comp) < 30:
        return np.nan

    # Build bins on the baseline distribution
    try:
        breakpoints = np.unique(np.nanpercentile(base, np.linspace(0, 100, n_bins + 1)))
    except Exception:
        return np.nan

    if len(breakpoints) < 3:
        return np.nan

    breakpoints[0]  -= 1e-8
    breakpoints[-1] += 1e-8

    base_counts, _ = np.histogram(base, bins=breakpoints)
    comp_counts, _ = np.histogram(comp, bins=breakpoints)

    base_pct = base_counts / base_counts.sum()
    comp_pct = comp_counts / comp_counts.sum()

    # Replace zeros to avoid log(0)
    eps = 1e-6
    base_pct = np.where(base_pct == 0, eps, base_pct)
    comp_pct = np.where(comp_pct == 0, eps, comp_pct)

    psi = np.sum((comp_pct - base_pct) * np.log(comp_pct / base_pct))
    return float(psi)


def psi_label(psi: float) -> str:
    if np.isnan(psi):          return "unknown"
    if psi < 0.10:             return "stable"
    if psi < 0.25:             return "moderate drift"
    return "significant drift"


# ─── Candidate numeric drift features ────────────────────────────────────────
NUMERIC_DRIFT_CANDIDATES = [
    "BRIDGE_AGE", "TRAFFIC_DENSITY", "TIME_SINCE_RECONSTRUCTION",
    "AGE_X_SCOUR", "AGE_TO_SPAN_RATIO", "DECK_TO_ROADWAY_RATIO",
    "ADT_GROWTH_RATIO", "INVENTORY_RATING_066", "OPERATING_RATING_064",
]
numeric_drift_features = [c for c in NUMERIC_DRIFT_CANDIDATES if c in df.columns]
print(f"Numeric drift features found: {len(numeric_drift_features)}")
print(numeric_drift_features)


In [ ]:
# ─── Compute PSI across years ─────────────────────────────────────────────────
if YEAR_COL and len(df[YEAR_COL].dropna().unique()) >= 2:
    years_sorted = sorted(df[YEAR_COL].dropna().unique())
    BASELINE_YEAR = years_sorted[0]
    baseline_df = df[df[YEAR_COL] == BASELINE_YEAR]

    psi_records = []
    for year in years_sorted[1:]:
        comp_df = df[df[YEAR_COL] == year]
        for feat in numeric_drift_features:
            psi = compute_psi(baseline_df[feat], comp_df[feat])
            psi_records.append({
                "feature":         feat,
                "baseline_year":   int(BASELINE_YEAR),
                "comparison_year": int(year),
                "psi":             psi,
                "drift_level":     psi_label(psi),
            })

    drift_df = pd.DataFrame(psi_records)
    print(f"\n{'='*55}")
    print(" NUMERIC DRIFT SUMMARY (PSI)")
    print(f"{'='*55}")
    print(drift_df.sort_values("psi", ascending=False).to_string(index=False))
    drift_df.to_csv("safespan_data_drift_results.csv", index=False)
    print("\n✅ Saved: safespan_data_drift_results.csv")

else:
    # ─── Single-year fallback: demonstrate PSI methodology ───────────────────
    print("⚠️  Only one year detected — demonstrating PSI on train/test cohorts.")
    print("    In production with multi-year data, PSI will compare year-over-year.")

    rng = np.random.default_rng(42)
    mask = rng.random(len(df)) < 0.5
    baseline_df = df[mask]
    comp_df     = df[~mask]
    BASELINE_YEAR = "Cohort A"

    psi_records = []
    for feat in numeric_drift_features:
        psi = compute_psi(baseline_df[feat], comp_df[feat])
        psi_records.append({
            "feature":         feat,
            "baseline_year":   "Cohort A",
            "comparison_year": "Cohort B",
            "psi":             psi,
            "drift_level":     psi_label(psi),
        })

    drift_df = pd.DataFrame(psi_records)
    print("\nPSI between random 50/50 splits (expect ~0 — methodology validation):")
    print(drift_df[["feature","psi","drift_level"]].sort_values("psi", ascending=False).to_string(index=False))
    drift_df.to_csv("safespan_data_drift_results.csv", index=False)
    print("\n✅ Saved: safespan_data_drift_results.csv")


In [ ]:
# ─── PSI visualisation ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Average PSI per feature (bar chart)
avg_psi = drift_df.groupby("feature")["psi"].mean().sort_values(ascending=False)

colors_bar = []
for val in avg_psi.values:
    if np.isnan(val):           colors_bar.append("#aaaaaa")
    elif val >= 0.25:           colors_bar.append("#e74c3c")
    elif val >= 0.10:           colors_bar.append("#e67e22")
    else:                       colors_bar.append("#27ae60")

axes[0].barh(avg_psi.index[::-1], avg_psi.values[::-1],
             color=colors_bar[::-1], edgecolor="black", linewidth=0.5)
axes[0].axvline(0.10, color="orange", linestyle="--", linewidth=1.2, label="Moderate (0.10)")
axes[0].axvline(0.25, color="red",    linestyle="--", linewidth=1.2, label="Significant (0.25)")
axes[0].set_xlabel("Population Stability Index (PSI)")
axes[0].set_title("Average PSI by Feature", fontweight="bold")
axes[0].legend(fontsize=9)

# Right: PSI trend by year (top 5 features)
if drift_df["comparison_year"].dtype in [float, int] or (
    drift_df["comparison_year"].nunique() > 1
    and drift_df["comparison_year"].iloc[0] != "Cohort B"
):
    top5_features = avg_psi.head(5).index.tolist()
    for feat in top5_features:
        sub = drift_df[drift_df["feature"] == feat].sort_values("comparison_year")
        axes[1].plot(sub["comparison_year"].astype(str), sub["psi"],
                     marker="o", label=feat, linewidth=1.8)
    axes[1].axhline(0.10, color="orange", linestyle="--", linewidth=1, alpha=0.7)
    axes[1].axhline(0.25, color="red",    linestyle="--", linewidth=1, alpha=0.7)
    axes[1].set_xlabel("Comparison Year")
    axes[1].set_ylabel("PSI")
    axes[1].set_title("PSI Trend Over Years — Top 5 Features", fontweight="bold")
    axes[1].legend(fontsize=8)
else:
    axes[1].set_visible(False)

plt.suptitle("Data Drift Analysis — PSI (Numeric Features)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/drift_psi_numeric.png", dpi=150, bbox_inches="tight")
plt.show()


### 2.2 — Categorical Feature Drift (Jensen-Shannon Distance)

Categorical distributions are compared using **Jensen-Shannon distance**,
a symmetric, bounded (0–1) measure of how different two distributions are.


In [ ]:
# ─── JS distance helper ──────────────────────────────────────────────────────
def compute_js_distance(base_series: pd.Series, comp_series: pd.Series) -> float:
    """Jensen-Shannon distance between two categorical distributions."""
    base = base_series.dropna()
    comp = comp_series.dropna()
    if len(base) < 10 or len(comp) < 10:
        return np.nan

    all_cats = list(set(base.unique()) | set(comp.unique()))
    p = np.array([base.value_counts().get(c, 0) for c in all_cats], dtype=float)
    q = np.array([comp.value_counts().get(c, 0) for c in all_cats], dtype=float)

    p /= p.sum()
    q /= q.sum()

    return float(jensenshannon(p, q))


def js_label(js: float) -> str:
    if np.isnan(js):    return "unknown"
    if js < 0.05:       return "low drift"
    if js < 0.15:       return "moderate drift"
    return "high drift"


# ─── Candidate categorical drift features ────────────────────────────────────
CATEGORICAL_DRIFT_CANDIDATES = [
    "SCOUR_CRITICAL_113", "STRUCTURE_KIND_043A",
    "DESIGN_LOAD_031",    "SERVICE_ON_042A",
    "STATE_CODE_001",
]
cat_drift_features = [c for c in CATEGORICAL_DRIFT_CANDIDATES if c in df.columns]
print(f"Categorical drift features found: {len(cat_drift_features)}")


In [ ]:
if cat_drift_features:
    if YEAR_COL and len(df[YEAR_COL].dropna().unique()) >= 2:
        years_sorted = sorted(df[YEAR_COL].dropna().unique())
        baseline_df_cat = df[df[YEAR_COL] == years_sorted[0]]
        js_records = []
        for year in years_sorted[1:]:
            comp_df_cat = df[df[YEAR_COL] == year]
            for feat in cat_drift_features:
                js = compute_js_distance(
                    baseline_df_cat[feat].astype(str),
                    comp_df_cat[feat].astype(str),
                )
                js_records.append({
                    "feature":         feat,
                    "baseline_year":   int(years_sorted[0]),
                    "comparison_year": int(year),
                    "js_distance":     js,
                    "drift_level":     js_label(js),
                })
    else:
        # Single-year: compare random halves
        rng = np.random.default_rng(42)
        mask = rng.random(len(df)) < 0.5
        js_records = []
        for feat in cat_drift_features:
            js = compute_js_distance(
                df[mask][feat].astype(str),
                df[~mask][feat].astype(str),
            )
            js_records.append({
                "feature":         feat,
                "baseline_year":   "Cohort A",
                "comparison_year": "Cohort B",
                "js_distance":     js,
                "drift_level":     js_label(js),
            })

    js_drift_df = pd.DataFrame(js_records)
    print("Categorical drift results:")
    print(js_drift_df.sort_values("js_distance", ascending=False).to_string(index=False))
    js_drift_df.to_csv("safespan_categorical_drift_results.csv", index=False)
    print("\n✅ Saved: safespan_categorical_drift_results.csv")

    # ─── Plot ────────────────────────────────────────────────────────────────
    avg_js = js_drift_df.groupby("feature")["js_distance"].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 4))
    colors_cat = ["#e74c3c" if v > 0.15 else "#e67e22" if v > 0.05 else "#27ae60"
                  for v in avg_js.values]
    ax.barh(avg_js.index[::-1], avg_js.values[::-1],
            color=colors_cat[::-1], edgecolor="black", linewidth=0.5)
    ax.axvline(0.05, color="orange", linestyle="--", linewidth=1.2, label="Moderate (0.05)")
    ax.axvline(0.15, color="red",    linestyle="--", linewidth=1.2, label="High (0.15)")
    ax.set_xlabel("Jensen-Shannon Distance")
    ax.set_title("Categorical Feature Drift (JS Distance)", fontweight="bold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("plots/drift_js_categorical.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("⚠️  No categorical drift features found in the dataset.")


---
## Section 3 — Target Drift Analysis

**Goal:** Check whether the distribution of `TARGET_CONDITION` (bridge condition
classes) changes across inspection years.

If the proportion of *Critical* or *Poor* bridges grows over time this signals
a systemic deterioration trend — important both for the model and for policy.


In [ ]:
if YEAR_COL and len(df[YEAR_COL].dropna().unique()) >= 2:
    years_sorted = sorted(df[YEAR_COL].dropna().unique())
    target_year = (
        df.groupby([YEAR_COL, TARGET_COL])
          .size()
          .reset_index(name="count")
    )
    target_year["pct"] = (
        target_year.groupby(YEAR_COL)["count"]
                   .transform(lambda x: x / x.sum() * 100)
    )
    target_year.to_csv("safespan_target_drift_by_year.csv", index=False)
    print("✅ Saved: safespan_target_drift_by_year.csv")

    # ─── Pivot for plotting ───────────────────────────────────────────────────
    pivot = (
        target_year.pivot(index=YEAR_COL, columns=TARGET_COL, values="pct")
                   .reindex(columns=CLASS_ORDER)
                   .fillna(0)
    )
    print("\nCondition distribution by year (%):")
    print(pivot.round(2).to_string())

    # Stacked bar chart
    fig, ax = plt.subplots(figsize=(12, 5))
    bottom = np.zeros(len(pivot))
    for cond in CLASS_ORDER:
        vals = pivot[cond].values if cond in pivot.columns else np.zeros(len(pivot))
        ax.bar(pivot.index.astype(str), vals, bottom=bottom,
               color=CONDITION_COLORS[cond], label=cond,
               edgecolor="white", linewidth=0.5)
        bottom += vals

    ax.set_xlabel("Inspection Year")
    ax.set_ylabel("Percentage (%)")
    ax.set_title("Bridge Condition Distribution by Year (Target Drift)",
                 fontweight="bold")
    ax.legend(title="Condition", bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("plots/target_drift_by_year.png", dpi=150, bbox_inches="tight")
    plt.show()

else:
    # Single-year: show current distribution as baseline
    print("⚠️  Only one inspection year available — showing current condition distribution.")
    print("    Target drift analysis requires multi-year data.")

    counts = df[TARGET_COL].value_counts().reindex(CLASS_ORDER)
    pct    = (counts / counts.sum() * 100).round(2)
    target_year = pd.DataFrame({"condition": CLASS_ORDER,
                                "count": counts.values,
                                "pct":   pct.values})
    target_year.to_csv("safespan_target_drift_by_year.csv", index=False)
    print("\nBaseline distribution:")
    print(target_year.to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(CLASS_ORDER, counts.values,
           color=[CONDITION_COLORS[c] for c in CLASS_ORDER],
           edgecolor="black", linewidth=0.7)
    ax.set_ylabel("Count")
    ax.set_title("Bridge Condition Distribution (Baseline Year)", fontweight="bold")
    plt.tight_layout()
    plt.savefig("plots/target_drift_by_year.png", dpi=150, bbox_inches="tight")
    plt.show()


---
## Section 4 — Model Performance Drift Analysis

**Goal:** Evaluate whether a model trained on historical data remains reliable
when evaluated on future inspection years.

### Temporal Split Strategy

| Split | Years | Purpose |
|-------|-------|---------|
| **Train** | ≤ 2022 | Fit the model |
| **Validation** | 2023 | Hyper-parameter / threshold tuning |
| **Test** | 2024, 2025 | Performance drift evaluation |

If the dataset spans fewer years, the notebook automatically falls back to a
70 / 15 / 15 chronological split.


In [ ]:
# ─── Prepare modeling dataset ─────────────────────────────────────────────────
# Label encoding
label_map = {c: i for i, c in enumerate(CLASS_ORDER)}  # Critical=0 … Good=3
df["TARGET_NUM"] = df[TARGET_COL].map(label_map)
df_model = df[df["TARGET_NUM"].notna()].copy()

# ─── Cap dataset size for modeling (keeps memory manageable) ──────────────────
# Drift and overview use the full df; modeling uses a stratified sample.
MAX_MODEL_ROWS = 200_000
if len(df_model) > MAX_MODEL_ROWS:
    df_model = (
        df_model.groupby("TARGET_NUM", group_keys=False)
                .apply(lambda g: g.sample(
                    frac=MAX_MODEL_ROWS / len(df_model), random_state=42
                ))
    )
    df_model = df_model.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"ℹ️  Sampled {len(df_model):,} rows for modeling (stratified, from {len(df):,} total)")

# ─── Leakage / ID columns to exclude from features ───────────────────────────
EXCLUDE_FROM_FEATURES = {
    "TARGET_CONDITION", "TARGET_NUM",
    # Raw NBI condition columns (direct leakage)
    "DECK_COND_058", "SUPERSTRUCTURE_COND_059",
    "SUBSTRUCTURE_COND_060", "CHANNEL_COND_061",
    "CULVERT_COND_062", "STRUCTURAL_EVAL_067",
    "DECK_GEOMETRY_EVAL_068", "UNDCLRENCE_EVAL_069",
    "POSTING_EVAL_070", "WATERWAY_EVAL_071",
    "APPR_ROAD_EVAL_072", "SUFFICIENCY_RATING",
    # Free-text / identifiers
    "STRUCTURE_NUMBER_008", "FEATURES_DESC_006A",
    "FACILITY_CARRIED_007", "LOCATION_009",
    "DATE_LAST_UPDATE", "DATE_OF_INSPECT_090",
    "OTHR_STATE_STRUC_NO_099", "REMARKS",
    "PROJ_NO", "PROJ_SUFFIX", "PROGRAM_CODE",
    "NBI_TYPE_OF_IMP",
}

feature_cols = [
    c for c in df_model.columns
    if c not in EXCLUDE_FROM_FEATURES
    and c != (YEAR_COL if YEAR_COL else "__none__")
    and df_model[c].nunique() > 1
]

print(f"Feature columns for modeling: {len(feature_cols)}")


In [ ]:
# ─── Temporal split logic ────────────────────────────────────────────────────
def make_temporal_splits(df_m, year_col):
    """Return (train, val, test_years_dict) based on available years."""
    if year_col is None or df_m[year_col].dropna().nunique() < 2:
        # No year info → random 70/15/15
        print("⚠️  No multi-year data — using random 70/15/15 split.")
        idx = np.arange(len(df_m))
        np.random.seed(42)
        np.random.shuffle(idx)
        n = len(idx)
        tr = df_m.iloc[idx[:int(0.70*n)]]
        va = df_m.iloc[idx[int(0.70*n):int(0.85*n)]]
        te = {"hold-out": df_m.iloc[idx[int(0.85*n):]]}
        return tr, va, te

    yrs = sorted(df_m[year_col].dropna().unique())
    if 2022 in yrs and any(y > 2022 for y in yrs):
        # Preferred temporal split
        train = df_m[df_m[year_col] <= 2022]
        val   = df_m[df_m[year_col] == 2023] if 2023 in yrs else df_m[df_m[year_col] == yrs[len(yrs)//2]]
        test_years = {int(y): df_m[df_m[year_col] == y]
                      for y in yrs if y >= 2024}
        if not test_years:
            test_years = {int(yrs[-1]): df_m[df_m[year_col] == yrs[-1]]}
    else:
        # Auto 70/15/15 on chronological order
        n_yr = len(yrs)
        tr_yrs = yrs[:max(1, int(0.70*n_yr))]
        va_yrs = yrs[len(tr_yrs):len(tr_yrs)+max(1, int(0.15*n_yr))]
        te_yrs = yrs[len(tr_yrs)+len(va_yrs):]
        if not te_yrs:
            te_yrs = yrs[-1:]
        train = df_m[df_m[year_col].isin(tr_yrs)]
        val   = df_m[df_m[year_col].isin(va_yrs)]
        test_years = {int(y): df_m[df_m[year_col] == y] for y in te_yrs}

    print(f"  Train:      {len(train):>8,} rows  (years {[int(y) for y in sorted(train[year_col].unique())]})")
    print(f"  Validation: {len(val):>8,} rows")
    for yr, te in test_years.items():
        print(f"  Test {yr}:  {len(te):>8,} rows")
    return train, val, test_years


print("=== Temporal Split ===")
train_df, val_df, test_years_dict = make_temporal_splits(df_model, YEAR_COL)


In [ ]:
# ─── Preprocessing pipeline ──────────────────────────────────────────────────
def build_feature_matrix(df_sub, feature_cols, fit_imputer=None, fit_encoder_cols=None):
    """
    Returns (X_array, fitted_imputer, encoder_cols).
    Pass fit_imputer=None to fit; pass the fitted object to transform-only.
    """
    X = df_sub[feature_cols].copy()

    # One-hot encode object columns
    obj_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    X = pd.get_dummies(X, columns=obj_cols, drop_first=True)

    # Align columns to training schema (for val / test sets)
    if fit_encoder_cols is not None:
        missing_cols = set(fit_encoder_cols) - set(X.columns)
        for mc in missing_cols:
            X[mc] = 0
        X = X[fit_encoder_cols]

    encoder_cols = X.columns.tolist()

    # Impute missing with median
    if fit_imputer is None:
        imp = SimpleImputer(strategy="median")
        X_arr = imp.fit_transform(X)
    else:
        imp = fit_imputer
        X_arr = imp.transform(X)

    return X_arr, imp, encoder_cols


X_train_arr, imputer_fitted, enc_cols = build_feature_matrix(train_df, feature_cols)
y_train = train_df["TARGET_NUM"].values.astype(int)

X_val_arr, _, _ = build_feature_matrix(val_df, feature_cols, imputer_fitted, enc_cols)
y_val = val_df["TARGET_NUM"].values.astype(int)

print(f"X_train shape: {X_train_arr.shape}  | y_train classes: {np.unique(y_train)}")
print(f"X_val   shape: {X_val_arr.shape}")


In [ ]:
# ─── Train the model ─────────────────────────────────────────────────────────
if LGBM_AVAILABLE:
    from sklearn.utils.class_weight import compute_sample_weight
    sw = compute_sample_weight("balanced", y_train)

    model = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=63,
        class_weight="balanced",
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train_arr, y_train, sample_weight=sw)
    MODEL_NAME = "LightGBM"
else:
    model = RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_train_arr, y_train)
    MODEL_NAME = "RandomForest"

print(f"✅ {MODEL_NAME} trained on {len(y_train):,} samples")


In [ ]:
# ─── Safety metric: Critical → Good error rate ───────────────────────────────
def critical_to_good_error_rate(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Fraction of truly Critical bridges (label=0) predicted as Good (label=3).
    This is the worst safety failure: a bridge in critical need is dismissed.
    Lower is better; target < 2%.
    """
    mask = y_true == 0   # Critical
    if mask.sum() == 0:
        return np.nan
    return float((y_pred[mask] == 3).sum() / mask.sum())


def evaluate_period(y_true, y_pred, period_label):
    """Compute and return a metrics dict for one evaluation period."""
    metrics = {
        "period":           period_label,
        "n_samples":        len(y_true),
        "macro_f1":         round(f1_score(y_true, y_pred, average="macro",    zero_division=0), 4),
        "weighted_f1":      round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "accuracy":         round(accuracy_score(y_true, y_pred), 4),
        "critical_recall":  round(f1_score(y_true, y_pred, labels=[0], average=None, zero_division=0)[0], 4),
        "critical_to_good": round(critical_to_good_error_rate(y_true, y_pred) or 0, 4),
    }
    return metrics


# Validation set
y_val_pred = model.predict(X_val_arr)
perf_records = [evaluate_period(y_val, y_val_pred, "Validation")]

# Future years
for yr, te_df in test_years_dict.items():
    if len(te_df) < 10:
        continue
    X_te_arr, _, _ = build_feature_matrix(te_df, feature_cols, imputer_fitted, enc_cols)
    y_te = te_df["TARGET_NUM"].values.astype(int)
    y_te_pred = model.predict(X_te_arr)
    perf_records.append(evaluate_period(y_te, y_te_pred, str(yr)))

perf_df = pd.DataFrame(perf_records)
perf_df.to_csv("safespan_model_performance_drift.csv", index=False)
print(perf_df.to_string(index=False))
print("\n✅ Saved: safespan_model_performance_drift.csv")


In [ ]:
# ─── Performance drift plots ─────────────────────────────────────────────────
if len(perf_df) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    metrics_to_plot = [
        ("macro_f1",         "Macro F1",             "#2980b9"),
        ("critical_recall",  "Critical Recall",       "#e74c3c"),
        ("critical_to_good", "Critical→Good Error %", "#8e44ad"),
    ]
    for ax, (col, label, color) in zip(axes, metrics_to_plot):
        vals = perf_df[col].values
        if col == "critical_to_good":
            vals = vals * 100   # show as percentage
        ax.plot(perf_df["period"], vals, marker="o", color=color, linewidth=2)
        ax.set_title(label, fontweight="bold")
        ax.set_xlabel("Evaluation Period")
        ax.set_ylabel("%" if col == "critical_to_good" else "Score")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", alpha=0.4)

    plt.suptitle(f"Model Performance Drift — {MODEL_NAME}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("plots/model_performance_drift.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("ℹ️  Only one evaluation period — need multi-year data to plot drift trend.")


### Interpretation

- **Macro F1 declining** across years → the model generalises less well to newer
  inspection data. This motivates periodic retraining.
- **Critical Recall declining** → the model misses an increasing fraction of
  bridges that are truly in Critical condition — a safety concern.
- **Critical → Good error rising** → more Critical bridges are misclassified as
  Good, the worst possible safety failure. Any value above 2% should trigger an
  immediate review of the model.


---
## Section 5 — Survival Analysis Dataset Construction

**Goal:** Reshape the bridge-year panel into one row per bridge for
time-to-deterioration (survival) analysis.

### Event definition

| Term | Meaning |
|------|---------|
| **Event** | Bridge enters *Poor* or *Critical* condition |
| **Origin** | First year where the bridge is *Good* or *Fair* |
| **Duration** | Years from origin to event (or to last observation if censored) |
| **Censored (event=0)** | Bridge never reaches Poor/Critical during study window |

### Why survival analysis for bridges?

Classification answers "What is this bridge's condition *today*?"
Survival analysis answers "**How long** until this bridge deteriorates?"
This time-based perspective directly supports *proactive* maintenance scheduling.


In [ ]:
# ─── Cross-sectional survival dataset ────────────────────────────────────────
# When only one inspection year is available we use BRIDGE_AGE as the time axis:
#   - Duration = BRIDGE_AGE (years the bridge has been in service)
#   - Event    = 1 if current condition is Poor or Critical
# This is a valid cross-sectional survival framing used in infrastructure literature.

POOR_CRITICAL = {"Critical", "Poor"}
GOOD_FAIR     = {"Good", "Fair"}

def build_survival_dataset_multi_year(df_m, bridge_col, year_col, target_col, feature_cols):
    """Panel data (multiple years per bridge) → one row per bridge."""
    df_sorted = df_m.sort_values([bridge_col, year_col])
    records = []

    for bridge_id, grp in df_sorted.groupby(bridge_col):
        grp = grp.reset_index(drop=True)

        # Find baseline: first year in Good/Fair
        gf_mask = grp[target_col].isin(GOOD_FAIR)
        if not gf_mask.any():
            continue

        baseline_idx  = gf_mask.idxmax()
        baseline_row  = grp.loc[baseline_idx]
        baseline_year = baseline_row[year_col]
        future        = grp[grp[year_col] > baseline_year]

        # Search for first Poor/Critical after baseline
        event_rows = future[future[target_col].isin(POOR_CRITICAL)]
        if len(event_rows) > 0:
            event_year = event_rows[year_col].min()
            duration   = event_year - baseline_year
            event      = 1
        else:
            duration = grp[year_col].max() - baseline_year
            event    = 0

        rec = {
            "bridge_id":     bridge_id,
            "baseline_year": baseline_year,
            "duration":      duration,
            "event":         event,
        }
        for fc in feature_cols:
            if fc in baseline_row.index:
                rec[fc] = baseline_row[fc]
        records.append(rec)

    return pd.DataFrame(records)


def build_survival_dataset_cross_sectional(df_m, target_col, feature_cols, bridge_col=None):
    """
    Cross-sectional survival dataset using BRIDGE_AGE as the time axis.
    Works with single inspection year.
    """
    if "BRIDGE_AGE" not in df_m.columns:
        raise ValueError("BRIDGE_AGE is required for cross-sectional survival analysis.")

    # Deduplicate the column list to avoid duplicate-column issues
    cols_wanted = list(dict.fromkeys(
        [target_col, "BRIDGE_AGE"]
        + [f for f in feature_cols if f in df_m.columns and f != target_col and f != "BRIDGE_AGE"]
    ))
    surv = df_m[cols_wanted].copy()
    if bridge_col and bridge_col in df_m.columns:
        surv["bridge_id"] = df_m[bridge_col].values

    surv["duration"] = surv["BRIDGE_AGE"].values.clip(0)  # numpy array avoids Series shape issues
    surv["event"]    = surv[target_col].isin(POOR_CRITICAL).astype(int)
    surv.drop(columns=[target_col], inplace=True)
    return surv


# ─── Choose approach based on available data ──────────────────────────────────
if (YEAR_COL and BRIDGE_COL
        and df_model[YEAR_COL].dropna().nunique() >= 2
        and df_model[BRIDGE_COL].dropna().nunique() > 1):
    print("Using multi-year panel approach for survival dataset…")
    surv_feat_cols = [c for c in numeric_drift_features if c in df_model.columns]
    survival_df = build_survival_dataset_multi_year(
        df_model, BRIDGE_COL, YEAR_COL, TARGET_COL, surv_feat_cols
    )
else:
    print("Using cross-sectional approach (single inspection year)…")
    surv_feat_cols = [c for c in numeric_drift_features if c in df_model.columns]
    survival_df = build_survival_dataset_cross_sectional(
        df_model, TARGET_COL, surv_feat_cols, BRIDGE_COL
    )

print(f"\nSurvival dataset shape: {survival_df.shape}")

# ──── Drop zero-duration rows (required for Cox model) ────────────────────────
n_zero_dur = (survival_df["duration"] <= 0).sum()
survival_df = survival_df[survival_df["duration"] > 0].copy()
print(f"Dropped {n_zero_dur:,} rows with duration ≤ 0")

# ──── Summary statistics ──────────────────────────────────────────────────────
n_total    = len(survival_df)
n_events   = survival_df["event"].sum()
n_censored = n_total - n_events
event_rate = n_events / n_total * 100
med_dur    = survival_df["duration"].median()

print(f"\n{'='*45}")
print(" SURVIVAL DATASET SUMMARY")
print(f"{'='*45}")
print(f"  Total bridges:        {n_total:>10,}")
print(f"  Events (Poor/Crit):   {n_events:>10,}  ({event_rate:.1f}%)")
print(f"  Censored:             {n_censored:>10,}  ({100-event_rate:.1f}%)")
print(f"  Median duration:      {med_dur:>10.1f} years")
print(f"{'='*45}")

survival_df.to_csv("safespan_survival_dataset.csv", index=False)
print("\n✅ Saved: safespan_survival_dataset.csv")


---
## Section 6 — Kaplan-Meier Survival Analysis

The **Kaplan-Meier estimator** is a non-parametric method that estimates the
survival function — the probability that a bridge remains in Good/Fair condition
beyond time *t*.

- **No assumptions** about the shape of the hazard function.
- Automatically handles **censored** observations (bridges that never deteriorate
  during the study window).
- **Log-rank test** compares KM curves between groups.


In [ ]:
if not LIFELINES_AVAILABLE:
    print("⚠️  lifelines is not installed. Skipping Kaplan-Meier analysis.")
    print("    Install with: pip install lifelines")
else:
    # ─── Overall KM curve ────────────────────────────────────────────────────
    kmf = KaplanMeierFitter()
    kmf.fit(
        durations=survival_df["duration"],
        event_observed=survival_df["event"],
        label="All Bridges",
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    kmf.plot_survival_function(ax=ax, ci_show=True, color="#2980b9", linewidth=2)

    ax.set_xlabel("Years Since Baseline Inspection", fontsize=12)
    ax.set_ylabel("P(Remaining in Good/Fair Condition)", fontsize=12)
    ax.set_title("Kaplan-Meier Survival Curve\nTime Until Poor/Critical Condition",
                 fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color="grey", linestyle="--", linewidth=1, alpha=0.7,
               label="50% survival")
    ax.legend()
    plt.tight_layout()
    plt.savefig("safespan_kaplan_meier_overall.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Median survival time: {kmf.median_survival_time_:.2f} years")


In [ ]:
if LIFELINES_AVAILABLE:
    # ─── KM by BRIDGE_AGE group ───────────────────────────────────────────────
    if "BRIDGE_AGE" in survival_df.columns:
        age_median = survival_df["BRIDGE_AGE"].median()
        survival_df["age_group"] = np.where(
            survival_df["BRIDGE_AGE"] > age_median, "High Age", "Low Age"
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        for grp, color in [("High Age", "#e74c3c"), ("Low Age", "#27ae60")]:
            sub = survival_df[survival_df["age_group"] == grp]
            kmf_g = KaplanMeierFitter()
            kmf_g.fit(sub["duration"], sub["event"], label=grp)
            kmf_g.plot_survival_function(ax=ax, ci_show=True, color=color, linewidth=2)

        # Log-rank test
        hi = survival_df[survival_df["age_group"] == "High Age"]
        lo = survival_df[survival_df["age_group"] == "Low Age"]
        lr = logrank_test(hi["duration"], lo["duration"],
                          hi["event"], lo["event"])
        ax.set_title(f"Kaplan-Meier by Bridge Age  (log-rank p = {lr.p_value:.4f})",
                     fontweight="bold")
        ax.set_xlabel("Duration (years)")
        ax.set_ylabel("P(Remaining in Good/Fair Condition)")
        ax.set_ylim(0, 1.05)
        plt.tight_layout()
        plt.savefig("safespan_km_by_age.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  Log-rank p-value (age group): {lr.p_value:.4f}")
    else:
        print("⚠️  BRIDGE_AGE not in survival dataset — skipping age-group KM plot.")


In [ ]:
if LIFELINES_AVAILABLE:
    # ─── KM by SCOUR_CRITICAL_113 ─────────────────────────────────────────────
    if "SCOUR_CRITICAL_113" in survival_df.columns and survival_df["SCOUR_CRITICAL_113"].dtype != object:
        survival_df["scour_group"] = np.where(
            survival_df["SCOUR_CRITICAL_113"].fillna(9) <= 3,
            "High Scour Risk (≤3)",
            "Low Scour Risk (>3)",
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        for grp, color in [("High Scour Risk (≤3)", "#e74c3c"),
                            ("Low Scour Risk (>3)",  "#3498db")]:
            sub = survival_df[survival_df["scour_group"] == grp]
            if len(sub) < 10:
                continue
            kmf_g = KaplanMeierFitter()
            kmf_g.fit(sub["duration"], sub["event"], label=grp)
            kmf_g.plot_survival_function(ax=ax, ci_show=True, color=color, linewidth=2)

        hi = survival_df[survival_df["scour_group"] == "High Scour Risk (≤3)"]
        lo = survival_df[survival_df["scour_group"] == "Low Scour Risk (>3)"]
        if len(hi) > 5 and len(lo) > 5:
            lr_s = logrank_test(hi["duration"], lo["duration"],
                                hi["event"], lo["event"])
            p_str = f"p = {lr_s.p_value:.4f}"
            print(f"  Log-rank p-value (scour group): {lr_s.p_value:.4f}")
        else:
            p_str = "n/a (insufficient data)"

        ax.set_title(f"Kaplan-Meier by Scour Risk  (log-rank {p_str})", fontweight="bold")
        ax.set_xlabel("Duration (years)")
        ax.set_ylabel("P(Remaining in Good/Fair Condition)")
        ax.set_ylim(0, 1.05)
        plt.tight_layout()
        plt.savefig("safespan_km_by_scour.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("⚠️  SCOUR_CRITICAL_113 not available — skipping scour KM plot.")


In [ ]:
if LIFELINES_AVAILABLE:
    # ─── KM by AGE_X_SCOUR ───────────────────────────────────────────────────
    if "AGE_X_SCOUR" in survival_df.columns:
        axs_median = survival_df["AGE_X_SCOUR"].median()
        survival_df["axs_group"] = np.where(
            survival_df["AGE_X_SCOUR"] > axs_median,
            "High Age×Scour", "Low Age×Scour"
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        for grp, color in [("High Age×Scour", "#8e44ad"), ("Low Age×Scour", "#16a085")]:
            sub = survival_df[survival_df["axs_group"] == grp]
            if len(sub) < 10:
                continue
            kmf_g = KaplanMeierFitter()
            kmf_g.fit(sub["duration"], sub["event"], label=grp)
            kmf_g.plot_survival_function(ax=ax, ci_show=True, color=color, linewidth=2)

        hi = survival_df[survival_df["axs_group"] == "High Age×Scour"]
        lo = survival_df[survival_df["axs_group"] == "Low Age×Scour"]
        if len(hi) > 5 and len(lo) > 5:
            lr_axs = logrank_test(hi["duration"], lo["duration"],
                                  hi["event"], lo["event"])
            p_str = f"p = {lr_axs.p_value:.4f}"
            print(f"  Log-rank p-value (AGE_X_SCOUR group): {lr_axs.p_value:.4f}")
        else:
            p_str = "n/a"

        ax.set_title(f"Kaplan-Meier by Age×Scour Interaction  (log-rank {p_str})",
                     fontweight="bold")
        ax.set_xlabel("Duration (years)")
        ax.set_ylabel("P(Remaining in Good/Fair Condition)")
        ax.set_ylim(0, 1.05)
        plt.tight_layout()
        plt.savefig("safespan_km_by_age_x_scour.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("⚠️  AGE_X_SCOUR not in survival dataset — skipping.")


---
## Section 7 — Cox Proportional Hazards Model

The **Cox PH model** estimates how each feature influences the *hazard rate* —
the instantaneous risk of deteriorating into Poor/Critical condition.

**Key output: Hazard Ratio (HR)**
- HR > 1 → feature *increases* deterioration risk
- HR < 1 → feature *decreases* deterioration risk (protective)
- HR = 1 → no effect

⚠️ **Caution:** The Cox model assumes proportional hazards (the ratio of hazard
between groups is constant over time). With only 7 observation years (2018–2025)
this assumption should be checked and results interpreted carefully.


In [ ]:
if not LIFELINES_AVAILABLE:
    print("⚠️  lifelines not installed — skipping Cox PH model.")
else:
    # ─── Candidate Cox features ───────────────────────────────────────────────
    COX_CANDIDATES = [
        "BRIDGE_AGE", "TRAFFIC_DENSITY", "TIME_SINCE_RECONSTRUCTION",
        "AGE_X_SCOUR", "INVENTORY_RATING_066", "OPERATING_RATING_064",
        "SCOUR_CRITICAL_113",
    ]
    cox_features = [
        c for c in COX_CANDIDATES
        if c in survival_df.columns
        and pd.api.types.is_numeric_dtype(survival_df[c])
    ]
    print(f"Cox features: {cox_features}")

    if len(cox_features) == 0:
        print("⚠️  No numeric Cox features available.")
    else:
        # Build Cox dataframe
        cox_df = survival_df[["duration", "event"] + cox_features].copy()
        cox_df.dropna(subset=["duration", "event"], inplace=True)

        # Fill feature NaN with median
        for fc in cox_features:
            cox_df[fc].fillna(cox_df[fc].median(), inplace=True)

        # Standardise features for coefficient comparability
        scaler_cox = StandardScaler()
        cox_df[cox_features] = scaler_cox.fit_transform(cox_df[cox_features])

        # ─── Fit Cox model ────────────────────────────────────────────────────
        cph = CoxPHFitter(penalizer=0.1)
        try:
            cph.fit(cox_df, duration_col="duration", event_col="event",
                    show_progress=False)
            print("\n=== Cox PH Model Summary ===")
            cph.print_summary(decimals=4)

            # Save summary
            cox_summary = cph.summary.reset_index()
            cox_summary.to_csv("safespan_cox_hazard_summary.csv", index=False)
            print("\n✅ Saved: safespan_cox_hazard_summary.csv")

        except Exception as exc:
            print(f"⚠️  Cox model fitting failed: {exc}")
            cox_summary = None
            cph = None


In [ ]:
if LIFELINES_AVAILABLE and len(cox_features) > 0 and "cph" in dir() and cph is not None:
    # ─── Hazard Ratio plot ────────────────────────────────────────────────────
    summary = cph.summary.copy()
    summary = summary.sort_values("exp(coef)", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 5))
    colors_hr = ["#e74c3c" if hr > 1 else "#27ae60" for hr in summary["exp(coef)"]]
    bars = ax.barh(summary.index, summary["exp(coef)"],
                   color=colors_hr, edgecolor="black", linewidth=0.6, alpha=0.85)

    # 95% CI whiskers
    for i, (feat, row) in enumerate(summary.iterrows()):
        lo = row.get("exp(coef) lower 95%", row["exp(coef)"])
        hi = row.get("exp(coef) upper 95%", row["exp(coef)"])
        ax.plot([lo, hi], [i, i], color="black", linewidth=1.5, solid_capstyle="round")

    ax.axvline(1.0, color="black", linestyle="--", linewidth=1.2, label="HR = 1 (no effect)")
    ax.set_xlabel("Hazard Ratio (exp[coef])", fontsize=11)
    ax.set_title("Cox PH — Feature Hazard Ratios\n"
                 "HR > 1: increases deterioration risk  |  HR < 1: protective",
                 fontweight="bold")
    ax.legend(fontsize=9)

    # Colour legend
    hi_patch = mpatches.Patch(color="#e74c3c", label="Higher risk (HR > 1)")
    lo_patch = mpatches.Patch(color="#27ae60", label="Lower risk (HR < 1)")
    ax.legend(handles=[hi_patch, lo_patch], fontsize=9)
    plt.tight_layout()
    plt.savefig("plots/cox_hazard_ratios.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ─── Proportional hazards assumption check ────────────────────────────────
    print("\n=== Proportional Hazards Assumption Check ===")
    print("(Schoenfeld residuals test — p < 0.05 suggests violation)")
    try:
        cph.check_assumptions(cox_df, p_value_threshold=0.05, show_plots=False)
    except Exception as exc:
        print(f"  ⚠️  Assumption check could not complete: {exc}")
        print("  Interpret Cox results with caution given the limited time window.")


### Cox Model Interpretation Notes

- **BRIDGE_AGE (HR > 1):** Older bridges have a significantly higher hazard of
  deteriorating — consistent with physical decay mechanisms.
- **SCOUR_CRITICAL_113 (HR > 1 when low):** Bridges with poor scour protection
  face accelerated deterioration, especially after flood events.
- **INVENTORY_RATING_066 / OPERATING_RATING_064 (HR < 1):** Higher load ratings
  are associated with better-maintained, more recently built bridges — protective.
- **AGE_X_SCOUR (HR > 1):** The interaction amplifies risk; old bridges with
  poor scour protection are at the highest risk.

**Limitation:** The Cox model uses only 2018–2025 data (7 years maximum).
Infrastructure survival analyses ideally use decades of panel data. These
results are best treated as an *exploratory extension* that demonstrates the
methodology and informs long-term monitoring strategy.


---
## Section 8 — Final Summary & Project Integration

### How these modules extend SafeSpan

```
SafeSpan v1 (Static Classifier)
├── Input:  NBI inspection features
└── Output: Condition class (Critical / Poor / Fair / Good)

SafeSpan v2 (Temporal Decision-Support Framework)
├── Data Drift Monitor    → Flag when feature distributions shift year-to-year
├── Performance Monitor   → Alert when model accuracy drops on new inspections
├── Survival Model        → Estimate time until each bridge deteriorates
└── Maintenance Scheduler → Prioritise bridges by predicted hazard × consequence
```

### Key findings summary


In [ ]:
print("=" * 60)
print("  SAFESPAN FINAL SUMMARY")
print("=" * 60)

# Data overview
print(f"\n1. DATA")
print(f"   Rows:       {len(df):,}")
if YEAR_COL:
    yrs = sorted(df[YEAR_COL].dropna().unique())
    print(f"   Years:      {[int(y) for y in yrs]}")
for cond in CLASS_ORDER:
    n   = (df[TARGET_COL] == cond).sum()
    pct = n / len(df) * 100
    print(f"   {cond:<10s}: {n:>10,}  ({pct:.2f}%)")

# Drift
print(f"\n2. DATA DRIFT (PSI)")
if "drift_df" in dir():
    hi_drift = drift_df[drift_df["drift_level"].isin(["moderate drift", "significant drift"])]
    if len(hi_drift) > 0:
        top = hi_drift.nlargest(3, "psi")[["feature","psi","drift_level"]].to_string(index=False)
        print(f"   Top drifting features:\n{top}")
    else:
        print("   No significant drift detected.")

# Model performance
print(f"\n3. MODEL PERFORMANCE DRIFT ({MODEL_NAME})")
print(perf_df[["period","macro_f1","critical_recall","critical_to_good"]].to_string(index=False))

# Survival
print(f"\n4. SURVIVAL ANALYSIS")
print(f"   Bridges analysed:  {n_total:,}")
print(f"   Events (Poor/Crit): {n_events:,}  ({event_rate:.1f}%)")
print(f"   Median survival:    {med_dur:.1f} years")
if LIFELINES_AVAILABLE and "kmf" in dir():
    print(f"   KM median survival: {kmf.median_survival_time_:.1f} years")

print("\n" + "=" * 60)
print("  OUTPUT FILES GENERATED")
print("=" * 60)
output_files = [
    "safespan_data_drift_results.csv",
    "safespan_categorical_drift_results.csv",
    "safespan_target_drift_by_year.csv",
    "safespan_model_performance_drift.csv",
    "safespan_survival_dataset.csv",
    "safespan_cox_hazard_summary.csv",
    "safespan_kaplan_meier_overall.png",
    "safespan_km_by_age.png",
    "safespan_km_by_scour.png",
    "safespan_km_by_age_x_scour.png",
    "plots/00_dataset_overview.png",
    "plots/drift_psi_numeric.png",
    "plots/drift_js_categorical.png",
    "plots/target_drift_by_year.png",
    "plots/model_performance_drift.png",
    "plots/cox_hazard_ratios.png",
]
for f in output_files:
    exists = "✅" if os.path.exists(f) else "  "
    print(f"  {exists} {f}")


In [ ]:
# ─── Combined dashboard figure ───────────────────────────────────────────────
# Assembles key plots into a single presentation-ready summary figure.
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Panel 1 — Class distribution
ax1 = fig.add_subplot(gs[0, 0])
counts = df[TARGET_COL].value_counts().reindex(CLASS_ORDER)
ax1.bar(CLASS_ORDER, counts.values,
        color=[CONDITION_COLORS[c] for c in CLASS_ORDER],
        edgecolor="black", linewidth=0.7)
ax1.set_title("Condition Distribution", fontweight="bold")
ax1.set_ylabel("Count")
ax1.tick_params(axis="x", rotation=20)

# Panel 2 — PSI summary
ax2 = fig.add_subplot(gs[0, 1])
if "avg_psi" in dir():
    ax2.barh(avg_psi.index[::-1][:8], avg_psi.values[::-1][:8],
             color="#3498db", edgecolor="black", linewidth=0.5)
    ax2.axvline(0.10, color="orange", linestyle="--", linewidth=1)
    ax2.axvline(0.25, color="red",    linestyle="--", linewidth=1)
    ax2.set_title("Avg PSI by Feature", fontweight="bold")
    ax2.set_xlabel("PSI")

# Panel 3 — Model perf drift
ax3 = fig.add_subplot(gs[0, 2])
if len(perf_df) >= 1:
    ax3.plot(perf_df["period"], perf_df["macro_f1"],
             marker="o", color="#2980b9", label="Macro F1", linewidth=2)
    ax3.plot(perf_df["period"], perf_df["critical_recall"],
             marker="s", color="#e74c3c", label="Critical Recall", linewidth=2)
    ax3.set_title("Model Performance Drift", fontweight="bold")
    ax3.set_ylabel("Score")
    ax3.set_ylim(0, 1)
    ax3.legend(fontsize=8)
    ax3.tick_params(axis="x", rotation=30)

# Panel 4 — KM curve (if available)
ax4 = fig.add_subplot(gs[1, 0])
if LIFELINES_AVAILABLE and "kmf" in dir():
    kmf.plot_survival_function(ax=ax4, ci_show=True, color="#2980b9", linewidth=2)
    ax4.set_title("KM Survival Curve", fontweight="bold")
    ax4.set_xlabel("Duration (years)")
    ax4.set_ylabel("P(Survival)")
    ax4.set_ylim(0, 1.05)
else:
    ax4.text(0.5, 0.5, "lifelines\nnot installed",
             ha="center", va="center", transform=ax4.transAxes, fontsize=12,
             color="grey")
    ax4.set_title("KM Survival Curve", fontweight="bold")

# Panel 5 — KM by age group (if available)
ax5 = fig.add_subplot(gs[1, 1])
if LIFELINES_AVAILABLE and "age_group" in survival_df.columns:
    for grp, color in [("High Age", "#e74c3c"), ("Low Age", "#27ae60")]:
        sub = survival_df[survival_df["age_group"] == grp]
        kmf_g2 = KaplanMeierFitter()
        kmf_g2.fit(sub["duration"], sub["event"], label=grp)
        kmf_g2.plot_survival_function(ax=ax5, ci_show=False, color=color, linewidth=2)
    ax5.set_title("KM by Bridge Age", fontweight="bold")
    ax5.set_xlabel("Duration (years)")
    ax5.set_ylim(0, 1.05)
    ax5.legend(fontsize=8)
else:
    ax5.set_visible(False)

# Panel 6 — Cox hazard ratios
ax6 = fig.add_subplot(gs[1, 2])
if LIFELINES_AVAILABLE and "cph" in dir() and cph is not None:
    summ = cph.summary[["exp(coef)"]].sort_values("exp(coef)", ascending=True)
    colors_hr2 = ["#e74c3c" if v > 1 else "#27ae60" for v in summ["exp(coef)"]]
    ax6.barh(summ.index, summ["exp(coef)"],
             color=colors_hr2, edgecolor="black", linewidth=0.5)
    ax6.axvline(1, color="black", linestyle="--", linewidth=1)
    ax6.set_title("Cox Hazard Ratios", fontweight="bold")
    ax6.set_xlabel("HR")
else:
    ax6.set_visible(False)

fig.suptitle(
    "SafeSpan Bridge Analytics — Final Dashboard\n"
    "Drift · Model Performance · Survival Analysis",
    fontsize=14, fontweight="bold", y=1.01,
)
plt.savefig("plots/safespan_final_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Dashboard saved: plots/safespan_final_dashboard.png")


---
## Conclusion

This notebook has implemented three analytical extensions to the SafeSpan
predictive maintenance framework:

### 1. Data Drift Monitoring
By computing the Population Stability Index (PSI) for numeric features and
Jensen-Shannon distance for categorical features across inspection years, we
can automatically detect when the bridge inventory distribution drifts away
from the training distribution. This is the early warning system that tells
engineers *when* to retrain the model.

### 2. Model Performance Drift
Using a temporal train/test split, we showed how to evaluate LightGBM (or
RandomForest) performance separately for each future inspection year. The
*Critical → Good error rate* is the safety-critical metric: any increase in
this rate means bridges in Critical condition are being overlooked.

### 3. Survival Analysis
By reformulating the problem as time-to-deterioration, we estimated:
- The **median time** until a bridge enters Poor/Critical condition (Kaplan-Meier)
- Which **features accelerate** deterioration (Cox Proportional Hazards)

Together, these extensions make SafeSpan a **proactive** maintenance tool,
shifting from answering *"What is the condition today?"* to
*"How long until we need to act, and for which bridges?"*

---
*SafeSpan — DATA 245 Final Project*
